---

# C4. Exercițiu individual: construirea unui mini-prompt de adnotare

În acest exercițiu construiești un prompt mic de adnotare pentru comentarii politice.
- Intelegem cum se construiește un prompt: rol, variabile, definiții, reguli și format JSON.
- Alegemdouă axe proprii sau două axe din curs și vei testa promptul pe 5 comentarii.


## Pasul 0 . Configurare

In [1]:
import os, json, re, random
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
# caută .env urcând din folderul curent

ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

# DeepSeek
deepseek_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)
DEEPSEEK_MODEL = "deepseek-chat"
# Gemini prin OpenAI-compatible API
gemini_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

GEMINI_MODEL = "gemini-2.5-flash-lite"
# alegem modelul pentru demo

USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Root project:", ROOT)
print("DeepSeek key:", os.getenv("DEEPSEEK_API_KEY") is not None)
print("Gemini key:", os.getenv("GEMINI_API_KEY") is not None)
print("Model folosit:", model_now)
print("OK")

Root project: d:\ADC\AN 2\18. AI Algorithms\AI_Project\echochamber-project-team-6
DeepSeek key: True
Gemini key: True
Model folosit: gemini-2.5-flash-lite
OK


## Corpus

In [2]:
import pandas as pd
import random

corpus = pd.read_json("../../data/cleaned/corpus_youtube_sample.jsonl", lines=True)

print(len(corpus), "comentarii")
print("Câmpuri:", list(corpus.columns))

for _, c in corpus.sample(3).iterrows():
    print(f"[{c['source_channel'][:30]}] {c['text'][:80]}")

420 comentarii
Câmpuri: ['id', 'source_channel', 'video_title', 'text']
[expertforum] BRAVO ! Baga le Rachete de mezzo raggio vecine la granita, sì apoi plangete ca N
[NicusorDanRO] FELICITĂRI Dle PREȘEDINTE DAN si LA MULȚI ANI ROMÂNIA !!! 👏🇷🇴🇷🇴🇷🇴🇪🇺🇪🇺🇪🇺
[DianaSosoacaOfficial] Nu mor că slujesc lui Satan și o duc foarte bine s-au adunat cu schismaticii și 


### Pasul 1 Alege două axe

Alege două axe pe care vrei să le codezi.
Poți folosi axe din curs:
- institutional
- legitimare
- epistemic
- geopolitic
- mobilizare
Sau poți propune axe proprii:
- media_distrust
- elite_blame
- religious_frame
- fear
- irony
- people_vs_elite
- anti_corruption
- national_identity
Condiție: fiecare axă trebuie să aibă valori clare.
Pentru acest exercițiu folosim o scală simplă:
0 = absent
1 = prezent


In [3]:
# modifica dupa preferinte

AXA_1 = "anti_corruption"
AXA_2 = "national_identity"

## Pasul 2 — Definește axele
Scrie mai jos, în propriile cuvinte, ce înseamnă fiecare axă.
Exemplu:
media_distrust = comentariul exprimă neîncredere în presă, jurnaliști, televiziuni sau media mainstream.
religious_frame = comentariul folosește limbaj religios pentru a interpreta politica.

In [4]:
AXA_1_DEFINITION = """
anti-corruption măsoară daca se discursul militează împotriva corpuției
0 = absent
1 = prezent
2 = dominant
"""
AXA_2_DEFINITION = """
national-identity măsoasră dacă se pune accent pe identitate națională.
0 = absent
1 = prezent
2 = dominant
"""

## Pasul 3 — Construiește mini-promptul
Promptul trebuie să conțină:
1. rolul modelului;
2. sarcina;
3. definițiile celor două axe;
4. regulile de codare;
5. formatul JSON.
Important:
- nu cere modelului să identifice direct „bula”;
- nu cere text liber;
- returnează doar JSON valid.

In [5]:
MINI_PROMPT = f"""
Ești un comentator politic pe un canal politic.
SARCINĂ:
Adnotează comentariul folosind două axe:
1. {AXA_1}
2. {AXA_2}
CÂMPURI:
target = ținta politică principală din comentariu
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
{AXA_1} = 0 / 1 / 2
{AXA_2} = 0 / 1 / 2
DEFINIȚII:
{AXA_1_DEFINITION}
{AXA_2_DEFINITION}
REGULI:
1. Codează doar ce apare în comentariu, titlu sau canal.
2. Nu inventa informații externe.
3. Dacă nu există target politic, folosește target="none" și stance="none".
4. Dacă textul este ironic, codează sensul intenționat, nu sensul literal.
5. Pentru axe: 0 = absent, 1 = prezent, 2 = dominant.
6. Nu atribui direct o bulă discursivă.
7. Returnează doar JSON valid.
FORMAT OUTPUT:
{{
  "target": "",
  "stance": "",
  "tone": "",
  "{AXA_1}": 0,
  "{AXA_2}": 0
}}
"""
print(MINI_PROMPT)


Ești un comentator politic pe un canal politic.
SARCINĂ:
Adnotează comentariul folosind două axe:
1. anti_corruption
2. national_identity
CÂMPURI:
target = ținta politică principală din comentariu
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
anti_corruption = 0 / 1 / 2
national_identity = 0 / 1 / 2
DEFINIȚII:

anti-corruption măsoară daca se discursul militează împotriva corpuției
0 = absent
1 = prezent
2 = dominant


national-identity măsoasră dacă se pune accent pe identitate națională.
0 = absent
1 = prezent
2 = dominant

REGULI:
1. Codează doar ce apare în comentariu, titlu sau canal.
2. Nu inventa informații externe.
3. Dacă nu există target politic, folosește target="none" și stance="none".
4. Dacă textul este ironic, codează sensul intenționat, nu sensul literal.
5. Pentru axe: 0 = absent, 1 = prezent, 2 = dominant.
6. Nu atribui direct o bulă discursivă.
7

## Pasul 4 — Alege 5 comentarii de test
Folosim un eșantion mic. Nu adnotăm tot corpusul.
Schimbă `random_state` ca să primești alte comentarii.

In [14]:
TESTS = corpus.sample(10)
TESTS[["id", "source_channel", "video_title", "text"]].head()

,id,source_channel,video_title,text
100,yt_KETrGGaXRpk_Ugzt4_wuK6WYCFeV09Z4AaABAg,turcescu111,Vizita stăpânului Zelenski la slugile preaple...,"Se comportă așa pentru că sunt marionete, slug..."
256,yt_yEuctxNb4O0_Ugx5WGcS2ytBm9ZVn9h4AaABAg,NicusorDanRO,🟢 LIVE - Întâlnire la Palatul Cotroceni cu mag...,Sunteți un președinte ilegitim un mincinos und...
392,yt_l_V0df1d8LE_Ugw0maoTWqNiT9W06Dh4AaABAg,@CălinGeorgescu-CanalulOficial,Călin Georgescu - Motivul anulării alegerilor ...,"La multi ani, cu sanatate si numai bucurii, Pr..."
363,yt_DKhN-ua4lyw_Ugy-Tn8ntNbT8dD0pM94AaABAg,georgesimionoficial,#gs #georgesimion #democratie #impreuna #prosp...,Haideți să ne luam țara înapoi CĂLIN GEORGESCU...
216,yt_joXkZDqGZQU_UgzU5EUL6tOzGzkdR754AaABAg,NicusorDanRO,🟢 Declarații de presă comune cu Președintele U...,"Foarte bine 👍, va rog sa trimiteti un camion d..."


## Pasul 5 — Rulează promptul pe cele 5 comentarii
Pentru fiecare comentariu:
1. trimitem canalul, titlul video și textul;
2. modelul returnează JSON;
3. citim rezultatul și verificăm dacă are sens.

In [18]:
USE_GEMINI = False
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Using:", model_now)

Using: deepseek-chat


In [19]:
def llm(system, user, max_tokens=700):
    response = client_now.chat.completions.create(
        model=model_now,
        temperature=0,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ]
    )
    return response.choices[0].message.content

In [20]:
results = []
for _, row in TESTS.iterrows():
    USER = f"""
CANAL:
{row.get("source_channel", "")}
TITLU VIDEO:
{row.get("video_title", "")}
COMENTARIU:
<<< {row["text"]} >>>
"""
    raw = llm(MINI_PROMPT, USER, max_tokens=300)
    print("=" * 80)
    print("COMENTARIU:")
    print(row["text"])
    print()
    print("OUTPUT MODEL:")
    print(raw)
    results.append({
        "id": row["id"],
        "text": row["text"],
        "model_output": raw
    })

COMENTARIU:
Se comportă așa pentru că sunt marionete, slugi, argați. Ei sunt ghidonați din străinătate, sunt PUȘI acolo unde sunt și așa își arată recunoștința față de stăpâni. Niște mizerabili, niste trădători, care ar trebui arestați, judecați și condamnați pentru trădarea țării și prăduirea avuției naționale.

OUTPUT MODEL:
{
  "target": "Zelenski și clasa politică românească",
  "stance": "anti",
  "tone": "acuzator",
  "anti_corruption": 1,
  "national_identity": 2
}
COMENTARIU:
Sunteți un președinte ilegitim un mincinos unde sunt dovezile pentru anularea alegerilor jigodie pusă de sistemul subteran!

OUTPUT MODEL:
{
  "target": "NicusorDan",
  "stance": "anti",
  "tone": "acuzator",
  "anti_corruption": 0,
  "national_identity": 0
}
COMENTARIU:
La multi ani, cu sanatate si numai bucurii, Presedintele nostru ales!❤️🤗🇷🇴💪

OUTPUT MODEL:
{
  "target": "Călin Georgescu",
  "stance": "pro",
  "tone": "afectiv",
  "anti_corruption": 0,
  "national_identity": 1
}
COMENTARIU:
Haideți să n

## Pasul 6 — Interpretare scurtă
Completează în notebook, în 3–5 rânduri:
- Ce două axe ai ales?
- De ce le-ai ales?
- Modelul a returnat JSON corect?
- Care a fost cea mai mare problemă?
- Ce ai schimba în prompt?

Am ales axa 1 = anti_corruption și axa 2 = national_identity pentru că mi se pare că sunt unele dintre cele mai frecvente axe aduse în discuție în general în astfel de comentarii. Modelul a returnat JSON corect. Cred ca daca ar fi să schimb ceva în prompt, aș adăuga poate definiții și mai clare și concise. 
